# Tool 5: Spread & Edge Analysis

## Purpose
Quantifies the bid-ask spread and available "edge" over time for each product.
This directly answers: **how much profit is available from market making,
and when?**

## Key Concepts
- **Spread** = ask_price_1 - bid_price_1 (the gap you'd pay to cross)
- **Buy Edge** = wall_mid - bid_price_1 (profit if you sell to best bid)
- **Sell Edge** = ask_price_1 - wall_mid (profit if you buy from best ask)
- Wide spread = more opportunity for market making
- Narrow spread = less room, need to be more precise

## What You'll See
1. Spread time series per product
2. Edge time series (buy side vs sell side)
3. Spread distribution histograms
4. Cross-product comparison to prioritize which to trade
5. Periods of unusually wide/narrow spreads

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DAY = -1  # -2, -1, or 0
FIG_WIDTH = 18
FIG_HEIGHT = 5

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_prices, compute_wall_mid, compute_spread,
    PRODUCTS, AVAILABLE_DAYS,
)

# Load both products
data = {}
for prod in PRODUCTS:
    p = load_prices(day=DAY, product=prod)
    p = compute_wall_mid(p)
    p = compute_spread(p)
    data[prod] = p

print(f"Loaded spread data for Day {DAY}")
for prod, p in data.items():
    print(f"  {prod}: {len(p)} snapshots, avg spread = {p['spread'].mean():.2f}")

In [ ]:
# ============================================================
# PLOT 1: Spread Over Time (Both Products)
# ============================================================

fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(FIG_WIDTH, FIG_HEIGHT * len(PRODUCTS)), sharex=True)
if len(PRODUCTS) == 1:
    axes = [axes]

for idx, prod in enumerate(PRODUCTS):
    p = data[prod]
    ax = axes[idx]
    spread = p["spread"].dropna()
    ts = p.loc[spread.index, "timestamp"]

    ax.plot(ts, spread, linewidth=0.8, alpha=0.7)
    ax.axhline(spread.mean(), color="red", linestyle="--", linewidth=1, alpha=0.7,
               label=f"Mean = {spread.mean():.1f}")
    ax.set_ylabel("Spread")
    ax.set_title(f"Bid-Ask Spread — {prod}", fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Edge Over Time (Buy Side vs Sell Side)
# ============================================================
# Buy edge = how much you'd profit buying at best bid relative to fair.
# Sell edge = how much you'd profit selling at best ask relative to fair.
# Asymmetry = directional opportunity.

fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(FIG_WIDTH, FIG_HEIGHT * len(PRODUCTS)), sharex=True)
if len(PRODUCTS) == 1:
    axes = [axes]

for idx, prod in enumerate(PRODUCTS):
    p = data[prod]
    ax = axes[idx]
    ts = p["timestamp"]

    ax.plot(ts, p["buy_edge"], label=f"Buy Edge (mean={p['buy_edge'].mean():.2f})", alpha=0.6)
    ax.plot(ts, p["sell_edge"], label=f"Sell Edge (mean={p['sell_edge'].mean():.2f})", alpha=0.6)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel("Edge")
    ax.set_title(f"Available Edge — {prod}", fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Spread Distribution Comparison
# ============================================================

fig, ax = plt.subplots(figsize=(FIG_WIDTH // 2, FIG_HEIGHT))

for prod in PRODUCTS:
    spread = data[prod]["spread"].dropna()
    ax.hist(spread, bins=30, alpha=0.5, label=f"{prod} (mean={spread.mean():.1f}, std={spread.std():.1f})")

ax.set_xlabel("Spread")
ax.set_ylabel("Count")
ax.set_title(f"Spread Distribution (Day {DAY})", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABLE: Cross-Day Spread Summary
# ============================================================
# Which product has the widest spread? Is it consistent across days?
# Wider spread = more profitable market making.

rows = []
for d in AVAILABLE_DAYS:
    for prod in PRODUCTS:
        p = compute_spread(compute_wall_mid(load_prices(day=d, product=prod)))
        s = p["spread"].dropna()
        rows.append({
            "Day": d, "Product": prod,
            "Mean Spread": s.mean(), "Std Spread": s.std(),
            "Min Spread": s.min(), "Max Spread": s.max(),
            "Mean Buy Edge": p["buy_edge"].mean(),
            "Mean Sell Edge": p["sell_edge"].mean(),
        })

summary = pd.DataFrame(rows)
print("Cross-day spread summary:\n")
print(summary.to_string(index=False, float_format="{:.2f}".format))